In [1]:
pip install -U pageindex openai groq python-dotenv

/Users/nupur.gulati/Downloads/study/VectorelessRAG/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()



print("page_index_key","✅" if os.getenv("PAGEINDEX_API_KEY") else "❌ Missing!")
print("groq_key","✅" if os.getenv("GROQ_API_KEY") else "❌ Missing!")

page_index_key ✅
groq_key ✅


In [4]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))
groq = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("✅ PageIndex Client ready")
print("✅ GROQ client ready")

✅ PageIndex Client ready
✅ GROQ client ready


## Section 2: Upload and index a pdf

#### Upload your PDF to the PageIndex cloud
#### PageIndex uses an LLM to read the document structure
#### Builds a hierarchical tree index (like a smart Table of Contents)
#### Returns a doc_id for all future operations

#### Why NO chunking?
#### Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the documents
#### natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [5]:
PDF_PATH = "./long_horizon_task_definition (1).pdf"

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

Uploading: ./long_horizon_task_definition (1).pdf
✅ Uploaded!
📋 Document ID: pi-cmqjrycn200zn01pa0yldp4xq
   (Save this ID — you'll use it throughout the notebook)


In [6]:
import time

In [7]:
print(f"Building tree index")
print(f"This runs once per document-doc index gets cached for reuse")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status",{status})

    if status == "completed":
        print(f"\n Index ready")
        break
    elif status == "failed":
        print(f"Processing failed. Check your pdf format")
        break

    time.sleep(5)


Building tree index
This runs once per document-doc index gets cached for reuse
Status {'completed'}

 Index ready


## Section 3: Inspect the Tree Structure

## Each node has:

#### node_id — unique ID used during retrieval
#### title — section heading
#### page_index — page number in original PDF
#### text — section summary (when node_summary=True)
#### nodes — child sections (nested)

In [8]:
# Fetch the full tree

import json


tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f" Top level sections",{len(pageindex_tree)})
print(f"Raw tree first node:")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

 Top level sections {1}
Raw tree first node:
{
  "title": "Long-Horizon Task Definition",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Long-Horizon Task Definition\n",
  "text": "# Long-Horizon Task Definition\n",
  "nodes": [
    {
      "title": "Part 1: Formal Definition",
      "node_id": "0001",
      "page_index": 1,
      "prefix_summary": "## Part 1: Formal Definition\n",
      "text": "## Part 1: Formal Definition\n",
      "nodes": [
        {
          "title": "TL;DR",
          "node_id": "0002",
          "page_index": 1,
          "summary": "### TL;DR\n\nA long-horizon task is a coherent work objective whose difficulty comes from sustained context intake, decision-making, dependent actions, qualitative judgment, and verification. We estimate it by the time a competent low-context human would need using normal workplace interfaces, not by the number of model tool calls.\n\n![img-0.jpeg](img-0.jpeg)\n",
          "text": "### TL;DR\n\nA long-horizon task

In [9]:
# Pretty print the full tree
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview"""
    for node in nodes:
        prefix = " " * indent + ("L_" if indent > 0 else "")
        page = node.get("page_index","?")
        print(f"{prefix} [{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"],indent +1)

print("Full document structure:\n")
print_tree(pageindex_tree)

Full document structure:

 [0000] Long-Horizon Task Definition (p.1)
 L_ [0001] Part 1: Formal Definition (p.1)
  L_ [0002] TL;DR (p.1)
  L_ [0003] Human Baseline (p.1)
  L_ [0004] Classification Tiers (p.2)
  L_ [0005] Guardrails (p.2)
  L_ [0006] Phase-Based Estimator (p.4)
  L_ [0007] Research Summary (p.4)
 L_ [0008] Part 2: Implementation Proposal (p.6)
  L_ [0009] Core Proposal (p.6)
  L_ [0010] Evidence Builder (p.7)
  L_ [0011] Variable Extraction (p.8)
  L_ [0012] Human-Time Judge (p.9)
  L_ [0013] Tool-Using LLM Assessor (p.9)
  L_ [0014] Review Use (p.10)


In [10]:
# Count total nodes
from platform import node

def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f" Total nodes in tree: {total}")
print("  Each node = one retrievable section of the document")

 Total nodes in tree: 15
  Each node = one retrievable section of the document


## Section 4: LLM Tree Search — The Core of PageIndex
#### This is where PageIndex fundamentally differs from vector RAG.

#### Vector RAG retrieval:
#### query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
#### Problem: finds what's similar, not what's relevant

#### PageIndex retrieval:
#### query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
#### Advantage: LLM understands document structure, context, and intent

#### The LLM acts like a human expert scanning a Table of Contents.

In [37]:
# LLM tree search functions
def llm_tree_search(query:str, tree:list, model:str = "qwen/qwen3-32b")->dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """

    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure(like a table of contents). Your task identify which
node ids most likely contain the answer to the query.
Think step by step about which sections are relevant.

Query: {query}

Document tree:
{json.dumps(compressed_tree, indent=2)}

REPLY ONLY in this exact json format:
{{
    "thinking": "<your step-by-step reasoning>",
    "node_list": ["node_id1","node_id2"]
}}
"""

    response = groq.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)




In [38]:
# Test with a sample query
query = "How do you classify long horizon task?"

print(f"Query:{query}")
result = llm_tree_search(query,pageindex_tree)
print(result.get("thinking","N/A"))
print()
print("Selected node ids",result.get("node_list",[]))


Query:How do you classify long horizon task?
The query asks about classifying long horizon tasks. The document's 'Long-Horizon Task Definition' (node 0000) directly addresses this. Its child nodes 'Classification Tiers' (0004) provides explicit tier categories for classification. 'Guardrails' (0005) clarifies boundaries for classification, while 'Phase-Based Estimator' (0006) offers methodology for estimating task duration. 'TL;DR' (0002) and 'Human Baseline' (0003) provide foundational context for understanding classification parameters. 'Research Summary' (0007) supports classification framework with academic references.

Selected node ids ['0001', '0004', '0005', '0006', '0007']


 Section 5: Full End-to-End RAG Pipeline
3 steps:

Tree Search → LLM picks relevant node_ids
Retrieve → Fetch the actual section content from those nodes
Generate → LLM writes a grounded answer with page citations
What makes this better than vector RAG:

Retrieved content has titles + page numbers (traceable)
LLM can cite exactly which section the answer comes from
No hallucination from irrelevant chunks

In [39]:
# Find nodes by Id

def find_nodes_by_id(tree:list, target_id:list)->list:
    """Recursively walk the tree and collect nodes matching target id's"""
    found =[]
    for node in tree:
        if node["node_id"] in target_id:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_id(node["nodes"], target_id=target_id))
    return found

In [40]:
# Generate answer from retrieved nodes

from multiprocessing import context

from urllib3 import response


def generate_answer(query:str, nodes:list, model:str="qwen/qwen3-32b")->str:
    """Takes retrieved nodes as context and generates the answer
    Instructs LLM to site section titles and page numbers"""
    if not nodes:
        return "No relevant sections found in the document"

    #Build context string from retrived nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' |  Page: {node.get('page_index','?')}]\n"
            f"{node.get('text','Content not available')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
    You are an expert document analyst.
    Amswer the question using only provided context.
    For every claim you make, cite the section page number titles in paranthesis.
    Be concise and precise.
    
    Question: {query}

    Context: {context}

    Answer:
    
    """
    response = groq.chat.completions.create(
        model = model,
        messages = [{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [41]:
# The complete vectorless RAG function
def vectoreless_rag(query:str, tree:list, verbose: bool =True)->str:
    """
    Full end to end RAG pipeline
    Step 1: LLM Tree Search -> find relevant node ids
    Step 2: Node retrieval -> fetches section content
    Step 3: Answer Generation -> produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list",[])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_id(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query,nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")
        
    return answer


In [42]:
# Run the full pipeline
answer = vectoreless_rag(query="What is an extended task in long horizon task definition?",
tree = pageindex_tree)

🔍 Query: What is an extended task in long horizon task definition?

🧠 Reasoning: The query asks for the definition of an 'extended task' in the context of long-horizon task definitions. The document's 'TL;DR' section (node 0002) in Part 1 provides a concise summary of long-horizon...
🎯 Retrieved node IDs: ['0002']
📄 Sections found: ['TL;DR']

📝 Answer:
<think>
Okay, let's see. The user is asking what an extended task is in the context of long-horizon task definitions. The provided context is from the 'TL;DR' section on Page 1. The main point there is that a long-horizon task involves sustained context intake, decision-making, dependent actions, qualitative judgment, and verification. The difficulty isn't measured by the number of model tool calls but by the time a competent low-context human would need using normal workplace interfaces.

Hmm, so the question is about an "extended task," but the context doesn't mention that term directly. Wait, maybe "extended task" is another way of re

In [43]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "How do you classify long horizon task?",
    "How do you classify extended task?"
]

for q in test_queries:
    print()
    ans = vectoreless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: How do you classify long horizon task?
A: <think>
Okay, let's tackle this question. The user is asking how to classify a long horizon task. I need to use only the provided context, which includes two sections: 'Classification Tiers' and 'Guardrails' on page 2.

First, looking at 'Classification Tiers', there's a table that categorizes tasks...
-------------------------------------------------------

Q: How do you classify extended task?
A: <think>
Okay, let's tackle this question. The user wants to know how to classify an extended task based on the provided context. First, I need to check the context given.

Looking at the first section titled 'Classification Tiers' on Page 2, there's a table that outlines different tiers based on est...
-------------------------------------------------------


 Section 6: Expert-Guided Retrieval
The killer feature no one talks about.

With vector RAG, injecting domain expertise requires fine-tuning the embedding model — expensive and time-consuming.

With PageIndex, you just add rules to the prompt:

"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.

In [44]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.

In [45]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [46]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# Krish Naik Academy | 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [47]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "qwen/qwen3-32b"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [48]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Criteria for long horizon task?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: Criteria for long horizon task?

── Without Expert Rules ──
Nodes: ['0002', '0004', '0005', '0006', '0007']

── With Expert Rules ──
Nodes: ['0004', '0005', '0006']
Reasoning: The query asks for the criteria of a long-horizon task. The document's 'Formal Definition' section (node 0001) contains 'Classification Tiers (0004)' which defines time-based task tiers and 'Guardrails (0005)' which outlines exclusion rules. These subsections directly address criteria for classifica


In [49]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_id(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Define long horizon task",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

<think>
Okay, I need to define what a long horizon task is based on the provided context. Let me look at the sections given.

First, there's a section titled "Long-Horizon Task Definition" on page 1. Then under "Part 1: Formal Definition" also on page 1, and a "TL;DR" subsection. The TL;DR says a long-horizon task is a coherent work objective with difficulty from sustained context intake, decision-making, dependent actions, qualitative judgment, and verification. It's estimated by the time a competent low-context human would need using normal workplace interfaces, not by the number of model tool calls.

So the key elements are the sustained aspects: context intake, decision-making, dependent actions, qualitative judgment, verification. Also, the estimation is based on human time, not tool calls. I need to make sure to mention all these points and cite the sections properly. The sections are on page 1, so I'll reference them as (Long-Horizon Task Definition, 1) and (TL;DR, 1). But since

 Section 7: Chat API — Zero LLM Setup
When to use this:

You don't want to manage OpenAI API calls yourself
You want a quick Q&A interface over your document
You're building a chat product and want PageIndex to handle everything
PageIndex provides its own LLM — you just pass a question and doc_id.

In [50]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here are the key findings from the document:

---

## Key Findings: Long-Horizon Task Definition

### 1. 📐 Core Definition
A **long-horizon task** is a coherent work objective whose difficulty stems from **sustained context intake, decision-making, dependent actions, qualitative judgment, and verification** — *not* from the number of model tool calls or transcript length.

---

### 2. ⏱️ Human-Baseline Measurement
Tasks are measured by the time a **competent, low-context human** (a professional with relevant skills but no special prior knowledge of the specific environment) would need using normal workplace interfaces. This is intentionally stricter than using a familiar expert and more realistic than comparing against LLM API-call speed.

---

### 3. 🏷️ Five Classification Tiers

| Human Time | Tier |
|---|---|
| < 1 hour | Routine |
| 1–4 hours | Extended |
| 4–10 hours | Long-horizon candidate |
| ≥ 10 hours | Deep long-horizon |
| Multi-day/week | Frontier produc

In [51]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "How do you define long horizon task??",
    "Waht is evidence builder?",
    "Explain guardrails as given in pdf document"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: How do you define long horizon task??
🤖 Assistant: Here's how the document defines a **long-horizon task**:

---

### Core Definition
> *"A long-horizon task is a **coherent work objective** whose difficulty comes from sustained context intake, decision-making, dependent actions, qualitative judgment, and verification."*

Crucially, it is measured by **the time a competent, low-context human would need** using normal workplace interfaces — **not**...
-------------------------------------------------------

👤 User: Waht is evidence builder?
🤖 Assistant: Here's what the **Evidence Builder** is:

---

### What It Is
The Evidence Builder is the **first stage of the two-stage classifier** proposed in Part 2. Its job is to **collect and package all relevant signals** from a task's static artifacts into a structured "evidence bundle" that the LLM judge can then use to estimate human completion time.

---

### Inputs
It takes these files as input:
```
i...
----------------------------

Section 8: Self-Hosted Open Source Option
Use this when:

You don't want to send documents to any cloud
You need full data privacy / on-prem deployment
You want to inspect or customize the tree-building logic
The open-source repo at https://github.com/VectifyAI/PageIndex lets you run the entire pipeline locally using your own OpenAI key.

What the CLI does:

Reads your PDF
Detects existing Table of Contents (if any)
Uses GPT-4o to build the hierarchical tree
Saves a document_name_pageindex.json alongside your PDF

In [52]:
# ── Clone the open-source repo ───────────────────────────────────────────────
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt

Cloning into 'PageIndex'...
remote: Enumerating objects: 1987, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 1987 (delta 108), reused 78 (delta 78), pack-reused 1793 (from 3)
Receiving objects: 100% (1987/1987), 24.06 MiB | 4.19 MiB/s, done.
Resolving deltas: 100% (1271/1271), done.
/Users/nupur.gulati/Downloads/study/VectorelessRAG/PageIndex
zsh:1: command not found: pip


In [53]:
# ── Create .env for self-hosted mode ─────────────────────────────────────────
# The local runner uses CHATGPT_API_KEY (not OPENAI_API_KEY)

import os
groq_key = os.getenv("GROQ_API_KEY", "")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={groq_key}\n")

print("✅ .env created for self-hosted mode")

✅ .env created for self-hosted mode


In [59]:
# ── Run PageIndex locally on a PDF ───────────────────────────────────────────
# Optional parameters you can customize:
#   --model                  OpenAI model (default: gpt-4o-2024-11-20)
#   --toc-check-pages        Pages to scan for existing TOC (default: 20)
#   --max-pages-per-node     Max pages per tree node (default: 10)
#   --if-add-node-summary    Include summaries in output (yes/no)

PDF_PATH = "./long_horizon_task_definition.pdf"   # ← change this

!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

Traceback (most recent call last):
  File "/Users/nupur.gulati/Downloads/study/VectorelessRAG/PageIndex/run_pageindex.py", line 4, in <module>
    from pageindex import *
  File "/Users/nupur.gulati/Downloads/study/VectorelessRAG/PageIndex/pageindex/__init__.py", line 1, in <module>
    from .page_index import *
  File "/Users/nupur.gulati/Downloads/study/VectorelessRAG/PageIndex/pageindex/page_index.py", line 7, in <module>
    from .utils import *
  File "/Users/nupur.gulati/Downloads/study/VectorelessRAG/PageIndex/pageindex/utils.py", line 1, in <module>
    import litellm
ModuleNotFoundError: No module named 'litellm'


In [57]:
!pip3 install litellm

Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none

In [60]:
# ── Load locally generated tree ──────────────────────────────────────────────
# Output is saved as: <your_pdf_name>_pageindex.json

import json

TREE_JSON_PATH = "/path/to/your/document_pageindex.json"  # ← change this

with open(TREE_JSON_PATH, "r") as f:
    local_tree = json.load(f)

print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

FileNotFoundError: [Errno 2] No such file or directory: '/path/to/your/document_pageindex.json'

In [ ]:
# ── Run the same RAG pipeline on the local tree ──────────────────────────────
# Everything from Sections 4–6 works identically with local trees

query  = "Summarize the executive summary section."
answer = vectoreless_rag(query, local_tree)

Section 9: Vector RAG vs PageIndex — Side-by-Side
Architecture Comparison
Aspect	Traditional Vector RAG	PageIndex (Vectorless RAG)
Document prep	Chunk into fixed pieces	Build hierarchical tree
Indexing	Embed each chunk	LLM reads structure
Storage	Vector database	JSON file
Query processing	Embed query → ANN search	LLM reasons over tree
What's retrieved	Flat anonymous chunks	Named sections + page refs
Explainability	❌ Opaque similarity score	✅ Traceable reasoning
Domain expertise	❌ Needs embedding fine-tune	✅ Add rules to prompt
Infrastructure	Pinecone / FAISS / ChromaDB	No vector DB needed
Best for	Short, diverse documents	Long, structured documents
FinanceBench accuracy	~80%	98.7%
When to use which
Use Vector RAG when:

Documents are short and varied (FAQs, product descriptions)
Semantic paraphrase matching is important
You need sub-second retrieval on millions of documents
Use PageIndex when:

Documents are long and professionally structured (reports, manuals, legal docs)
You need traceable, cited answers
Domain expertise should guide retrieval
You want to avoid vector DB infrastructure

In [61]:
# ── Quick comparison demo ────────────────────────────────────────────────────
# Show how the same query retrieves differently

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What is guardrails in pdf doc?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context
""")

print("=" * 55)
print("PAGEINDEX approach (actual):")
print("=" * 55)
print("""
result = llm_tree_search("What is guardrails in pdf doc?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number
""")

VECTOR RAG approach (conceptual):

query_vec = embed_model.encode("What is guardrails in pdf doc?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context

PAGEINDEX approach (actual):

result = llm_tree_search("What is guardrails in pdf doc?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number



In [62]:
# ── Delete document from cloud ───────────────────────────────────────────────
# WARNING: This permanently deletes the tree index.
# Comment this out if you want to reuse the doc_id later.

# pi_client.delete_document(doc_id)
# print(f"🗑️ Deleted document: {doc_id}")
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")

ℹ️ Deletion commented out — uncomment when you're done with this doc_id


Summary
You've now built a complete Vectorless RAG system with PageIndex.

What you built:
llm_tree_search() — LLM reasons over document tree to find relevant nodes
find_nodes_by_ids() — Retrieve actual section content from tree
generate_answer() — LLM produces cited, grounded answers
vectorless_rag() — Full pipeline combining all 3 steps
expert_rag() — Domain-guided retrieval without any fine-tuning
Chat API — Zero-setup document Q&A
Key takeaways:
Similarity ≠ Relevance — the fundamental flaw of vector search
Tree-based reasoning gives you traceable, accurate, explainable retrieval
Domain expertise injection is just prompt engineering — no model training needed
98.7% on FinanceBench vs ~80% for vector RAG